# Image Classification - Improved Solution
## WideResNet + Mixup/CutMix + TTA + Label Smoothing

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
import os
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
import random

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
# ===================== PATHS =====================
# Для локального запуска раскомментируй и поправь пути:
# img_dir = '/Users/amirhamidullin/Downloads/bhw1/trainval'
# test_dir = '/Users/amirhamidullin/Downloads/bhw1/test'
# labels_file = '/Users/amirhamidullin/Downloads/bhw1/labels.csv'
# submission_file = '/Users/amirhamidullin/Downloads/bhw1/sample_submission.csv'

# Для Kaggle/Colab:
img_dir = 'trainval'
test_dir = 'test'
labels_file = 'labels.csv'
submission_file = 'sample_submission.csv'

# ===================== CONFIG =====================
NUM_CLASSES = 200
IMG_SIZE = 32  # Можно 40, но 32 стандарт для WideResNet
BATCH_SIZE = 128
NUM_EPOCHS = 200
LR = 0.1
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
CUTMIX_PROB = 0.5
TTA_TRANSFORMS = 50  # Количество аугментаций при TTA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Dataset

In [3]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_df, img_dir, transform=None):
        self.img_labels = annotations_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.img_labels.iloc[idx, 1]
        
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# ===================== TRANSFORMS =====================
# Вычисляем mean/std по датасету (или используем стандартные)
MEAN = [0.4914, 0.4822, 0.4465]  # CIFAR-like
STD = [0.2470, 0.2435, 0.2616]

# Train transform - сильные аугментации
train_transform = T.Compose([
    T.ToImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(IMG_SIZE, padding=4),
    T.TrivialAugmentWide(),  # Автоматические аугментации
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=MEAN, std=STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.2)),  # Cutout-like
])

# Test transform - без аугментаций
test_transform = T.Compose([
    T.ToImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=MEAN, std=STD),
])

# TTA transform - для Test Time Augmentation
tta_transform = T.Compose([
    T.ToImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(IMG_SIZE, padding=4),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=MEAN, std=STD),
])

In [6]:
img_dir = '/Users/amirhamidullin/Downloads/bhw1/trainval'
test_dir = '/Users/amirhamidullin/Downloads/bhw1/test'
labels_file = '/Users/amirhamidullin/Downloads/bhw1/labels.csv'
submission_file = '/Users/amirhamidullin/Downloads/bhw1/sample_submission.csv'

In [7]:
# Load data
data = pd.read_csv(labels_file)
df_train, df_valid = train_test_split(
    data, test_size=0.1, random_state=SEED, stratify=data.Category
)
df_test = pd.read_csv(submission_file)

print(f"Train size: {len(df_train)}")
print(f"Val size: {len(df_valid)}")
print(f"Test size: {len(df_test)}")

dataset_train = CustomImageDataset(df_train, img_dir, transform=train_transform)
dataset_val = CustomImageDataset(df_valid, img_dir, transform=test_transform)
dataset_test = CustomImageDataset(df_test, test_dir, transform=test_transform)

# Для TTA нужен отдельный датасет
dataset_test_tta = CustomImageDataset(df_test, test_dir, transform=tta_transform)

train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

Train size: 90000
Val size: 10000
Test size: 10000


## WideResNet Architecture

In [8]:
# ===================== WIDE RESNET =====================
# Адаптирован для маленьких изображений (без агрессивного downsampling)

class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropout_rate=0.3):
        super(BasicBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.dropout = nn.Dropout(p=dropout_rate)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != out_planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)
            )

    def forward(self, x):
        out = self.dropout(self.conv1(F.relu(self.bn1(x))))
        out = self.conv2(F.relu(self.bn2(out)))
        out += self.shortcut(x)
        return out


class WideResNet(nn.Module):
    """WideResNet для маленьких изображений (CIFAR-style)
    
    Args:
        depth: общая глубина сети (должна быть 6n+4)
        widen_factor: множитель ширины каналов
        num_classes: количество классов
        dropout_rate: вероятность dropout
    
    Популярные конфигурации:
        WRN-16-4: depth=16, widen_factor=4 (~2.7M params) - быстрый
        WRN-28-10: depth=28, widen_factor=10 (~36M params) - лучшая точность
        WRN-40-4: depth=40, widen_factor=4 (~8.9M params) - баланс
    """
    def __init__(self, depth=28, widen_factor=10, num_classes=200, dropout_rate=0.3):
        super(WideResNet, self).__init__()
        self.in_planes = 16
        
        assert (depth - 4) % 6 == 0, 'Depth should be 6n+4'
        n = (depth - 4) // 6
        k = widen_factor
        
        nStages = [16, 16*k, 32*k, 64*k]
        
        # Первый conv без downsampling (важно для маленьких картинок!)
        self.conv1 = nn.Conv2d(3, nStages[0], kernel_size=3, stride=1, padding=1, bias=False)
        
        self.layer1 = self._make_layer(BasicBlock, nStages[1], n, stride=1, dropout_rate=dropout_rate)
        self.layer2 = self._make_layer(BasicBlock, nStages[2], n, stride=2, dropout_rate=dropout_rate)
        self.layer3 = self._make_layer(BasicBlock, nStages[3], n, stride=2, dropout_rate=dropout_rate)
        
        self.bn = nn.BatchNorm2d(nStages[3])
        self.linear = nn.Linear(nStages[3], num_classes)
        
        # Инициализация весов
        self._init_weights()

    def _make_layer(self, block, planes, num_blocks, stride, dropout_rate):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride, dropout_rate))
            self.in_planes = planes
        return nn.Sequential(*layers)
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.relu(self.bn(out))
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

In [9]:
# Создаём модель
# Для быстрых экспериментов: WRN-16-8 или WRN-28-4
# Для лучшей точности: WRN-28-10

model = WideResNet(depth=28, widen_factor=10, num_classes=NUM_CLASSES, dropout_rate=0.3).to(device)

# Подсчёт параметров
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")

Model parameters: 36,600,984


## Mixup & CutMix

In [10]:
# ===================== MIXUP =====================
def mixup_data(x, y, alpha=0.2):
    """Mixup: смешивает два изображения и их метки
    
    x_mixed = lambda * x_i + (1 - lambda) * x_j
    y_mixed = lambda * y_i + (1 - lambda) * y_j
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam


# ===================== CUTMIX =====================
def rand_bbox(size, lam):
    """Генерирует случайный bbox для CutMix"""
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    # Центр bbox
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2


def cutmix_data(x, y, alpha=1.0):
    """CutMix: вырезает патч из одного изображения и вставляет в другое"""
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    y_a, y_b = y, y[index]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    
    # Корректируем lambda по реальной площади
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))
    
    return x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss для Mixup/CutMix"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## Training

In [ ]:
# ===================== TRAINING UTILS =====================
sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12})

def plot_losses(train_losses, test_losses, train_accs, test_accs):
    clear_output(wait=True)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    
    axs[0].plot(train_losses, label='train')
    axs[0].plot(test_losses, label='val')
    axs[0].set_xlabel('epoch')
    axs[0].set_ylabel('loss')
    axs[0].legend()
    axs[0].set_title('Loss')
    
    axs[1].plot(train_accs, label='train')
    axs[1].plot(test_accs, label='val')
    axs[1].set_xlabel('epoch')
    axs[1].set_ylabel('accuracy')
    axs[1].legend()
    axs[1].set_title(f'Accuracy (best val: {max(test_accs):.4f})')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def training_epoch(model, optimizer, criterion, train_loader, use_mixup=True, use_cutmix=True):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for batch, labels in tqdm(train_loader, desc='Training', leave=False):
        batch = batch.to(device)
        labels = labels.to(device)
        
        # Выбираем: Mixup, CutMix или обычный batch
        r = np.random.rand()
        if use_cutmix and r < CUTMIX_PROB:
            batch, labels_a, labels_b, lam = cutmix_data(batch, labels, CUTMIX_ALPHA)
            use_mix = True
        elif use_mixup and r < CUTMIX_PROB + 0.3:  # 30% шанс mixup
            batch, labels_a, labels_b, lam = mixup_data(batch, labels, MIXUP_ALPHA)
            use_mix = True
        else:
            use_mix = False
        
        optimizer.zero_grad()
        logits = model(batch)
        
        if use_mix:
            loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
        else:
            loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch.size(0)
        _, predicted = logits.max(1)
        total += labels.size(0)
        
        if use_mix:
            correct += (lam * (predicted == labels_a).float() + 
                       (1 - lam) * (predicted == labels_b).float()).sum().item()
        else:
            correct += predicted.eq(labels).sum().item()
    
    return train_loss / total, correct / total


@torch.no_grad()
def validation_epoch(model, criterion, val_loader):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    for batch, labels in tqdm(val_loader, desc='Validating', leave=False):
        batch = batch.to(device)
        labels = labels.to(device)
        
        logits = model(batch)
        loss = criterion(logits, labels)
        
        val_loss += loss.item() * batch.size(0)
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return val_loss / total, correct / total

In [ ]:
def train(model, optimizer, scheduler, criterion, train_loader, val_loader, num_epochs, warmup_epochs=5):
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    best_acc = 0.0
    
    # Warmup LR
    warmup_lr = LR / warmup_epochs
    
    for epoch in range(1, num_epochs + 1):
        # Warmup
        if epoch <= warmup_epochs:
            for param_group in optimizer.param_groups:
                param_group['lr'] = warmup_lr * epoch
        
        train_loss, train_acc = training_epoch(model, optimizer, criterion, train_loader)
        val_loss, val_acc = validation_epoch(model, criterion, val_loader)
        
        if epoch > warmup_epochs and scheduler is not None:
            scheduler.step()
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
        
        plot_losses(train_losses, val_losses, train_accs, val_accs)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch}/{num_epochs} | LR: {current_lr:.6f} | '
              f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Best: {best_acc:.4f}')
    
    return train_losses, val_losses, train_accs, val_accs

In [ ]:
# ===================== SETUP TRAINING =====================
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

# Cosine Annealing - плавное уменьшение LR
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=NUM_EPOCHS - 5,  # минус warmup
    eta_min=1e-6
)

print(f"Training config:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LR}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Mixup alpha: {MIXUP_ALPHA}")
print(f"  CutMix alpha: {CUTMIX_ALPHA}")

In [ ]:
# ===================== TRAIN =====================
train_losses, val_losses, train_accs, val_accs = train(
    model, optimizer, scheduler, criterion, 
    train_loader, val_loader, NUM_EPOCHS,
    warmup_epochs=5
)

## Test Time Augmentation (TTA)

In [ ]:
# ===================== TTA - КЛЮЧЕВАЯ ШТУКА ДЛЯ BOOST =====================
# Идея: для каждой тестовой картинки делаем N аугментированных версий,
# получаем N предсказаний, усредняем вероятности

@torch.no_grad()
def predict_with_tta(model, test_df, img_dir, n_augments=50):
    """Test Time Augmentation
    
    Для каждой картинки:
    1. Делаем n_augments аугментированных версий
    2. Получаем предсказания для каждой
    3. Усредняем logits/probabilities
    4. Берём argmax
    """
    model.eval()
    predictions = []
    
    for idx in tqdm(range(len(test_df)), desc='TTA Prediction'):
        img_path = os.path.join(img_dir, test_df.iloc[idx, 0])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Собираем logits от всех аугментаций
        all_logits = []
        
        # Оригинальное изображение (без аугментации)
        img_tensor = test_transform(image).unsqueeze(0).to(device)
        logits = model(img_tensor)
        all_logits.append(logits)
        
        # Горизонтальный flip
        img_flipped = cv2.flip(image, 1)
        img_tensor = test_transform(img_flipped).unsqueeze(0).to(device)
        logits = model(img_tensor)
        all_logits.append(logits)
        
        # Аугментированные версии
        for _ in range(n_augments - 2):
            img_tensor = tta_transform(image).unsqueeze(0).to(device)
            logits = model(img_tensor)
            all_logits.append(logits)
        
        # Усредняем
        avg_logits = torch.mean(torch.cat(all_logits, dim=0), dim=0)
        pred = avg_logits.argmax().item()
        predictions.append(pred)
    
    return predictions


# Простой predict без TTA (для сравнения)
@torch.no_grad()
def predict_simple(model, loader):
    model.eval()
    predictions = []
    for images, _ in tqdm(loader, desc='Predicting'):
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1)
        predictions.extend(preds.cpu().numpy())
    return predictions

In [ ]:
# ===================== LOAD BEST MODEL & PREDICT =====================
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Без TTA (быстро, для проверки)
print("Predicting without TTA...")
predictions_simple = predict_simple(model, test_loader)

# С TTA (медленнее, но точнее)
print(f"\nPredicting with TTA ({TTA_TRANSFORMS} augments per image)...")
predictions_tta = predict_with_tta(model, df_test, test_dir, n_augments=TTA_TRANSFORMS)

In [ ]:
# ===================== SAVE SUBMISSIONS =====================
# Без TTA
submission_simple = pd.read_csv(submission_file)
submission_simple['Category'] = predictions_simple
submission_simple.to_csv('submission_no_tta.csv', index=False)
print("Saved: submission_no_tta.csv")

# С TTA
submission_tta = pd.read_csv(submission_file)
submission_tta['Category'] = predictions_tta
submission_tta.to_csv('labels_test.csv', index=False)  # Имя по заданию
print("Saved: labels_test.csv (with TTA)")

In [ ]:
# ===================== FINAL VALIDATION =====================
print(f"\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Best validation accuracy: {max(val_accs):.4f} ({max(val_accs)*100:.2f}%)")
print(f"Final validation accuracy: {val_accs[-1]:.4f} ({val_accs[-1]*100:.2f}%)")
print(f"\nSubmissions saved:")
print(f"  - submission_no_tta.csv (without TTA)")
print(f"  - labels_test.csv (with TTA - USE THIS!)")